# Module 01: Data Exploration and Visualization

Exploratory data analysis of the Credit Card Fraud Detection dataset, analyzing distributions, class imbalance, correlations, and key fraud indicators.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
try:
    credit_card_data = pd.read_csv("../creditcard.csv")
    print("Data loaded successfully from parent directory.")
except FileNotFoundError:
    try:
        credit_card_data = pd.read_csv("creditcard.csv")
        print("Data loaded successfully from current directory.")
    except FileNotFoundError:
        print("Error: creditcard.csv not found. Please verify the file path.")

In [ ]:
# Dataset information
print("Dataset Summary:")
print(credit_card_data.info())

In [ ]:
# First 5 rows of the dataset
print("Top 5 records:")
credit_card_data.head()

In [ ]:
# Last 5 rows of the dataset
print("Bottom 5 records:")
credit_card_data.tail()

In [ ]:
# Dataset dimensions
print(f"Total Rows: {credit_card_data.shape[0]:,}")
print(f"Total Columns: {credit_card_data.shape[1]}")

In [ ]:
# Statistical summary
credit_card_data.describe().round(4)

In [ ]:
# Check for missing values and duplicates
missing_vals = credit_card_data.isnull().sum().sum()
duplicate_rows = credit_card_data.duplicated().sum()
print(f"Total Missing Values: {missing_vals}")
print(f"Total Duplicate Rows: {duplicate_rows}")

## Target Variable Analysis: Class Distribution

The `Class` feature is our target:
- `0`: Legitimate transaction
- `1`: Fraudulent transaction

In [ ]:
# Class counts and proportions
class_counts = credit_card_data['Class'].value_counts()
class_percentages = credit_card_data['Class'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(f"Normal (0): {class_counts[0]:,} ({class_percentages[0]:.3f}%)")
print(f"Fraud  (1): {class_counts[1]:,} ({class_percentages[1]:.3f}%)")
print(f"Imbalance Ratio: 1 fraud per {int(class_counts[0] / class_counts[1])} normal transactions")

In [ ]:
# Visualize class imbalance
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.countplot(x='Class', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=ax[0])
ax[0].set_title('Transaction Class Count (0 = Normal, 1 = Fraud)', fontsize=13, fontweight='bold')
ax[0].set_xticklabels(['Normal (0)', 'Fraud (1)'])
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='center', xytext=(0, 6), textcoords='offset points')

# Pie chart
ax[1].pie(class_counts, labels=['Normal', 'Fraud'], autopct='%1.2f%%',
          colors=['#2b5c8f', '#d9534f'], explode=[0, 0.2], shadow=True, startangle=45)
ax[1].set_title('Transaction Class Proportions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## Transaction Amount Analysis

Comparing transaction amounts for fraudulent vs normal transactions.

In [ ]:
# Descriptive stats for Amount by Class
normal_amounts = credit_card_data[credit_card_data['Class'] == 0]['Amount']
fraud_amounts = credit_card_data[credit_card_data['Class'] == 1]['Amount']

print("Normal Transactions Amount Stats:")
print(normal_amounts.describe().round(2))
print("\nFraud Transactions Amount Stats:")
print(fraud_amounts.describe().round(2))

In [ ]:
# Visualizing Amount Distribution
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(normal_amounts, bins=50, ax=ax[0], color='#2b5c8f', kde=True)
ax[0].set_title('Distribution of Normal Transaction Amounts', fontsize=12, fontweight='bold')
ax[0].set_yscale('log')
ax[0].set_xlabel('Amount')

sns.histplot(fraud_amounts, bins=50, ax=ax[1], color='#d9534f', kde=True)
ax[1].set_title('Distribution of Fraud Transaction Amounts', fontsize=12, fontweight='bold')
ax[1].set_yscale('log')
ax[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

## Transaction Time Analysis

`Time` represents the seconds elapsed since the first transaction in the dataset.

In [ ]:
# Visualizing transaction time distribution
fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sns.histplot(credit_card_data[credit_card_data['Class'] == 0]['Time'] / 3600, bins=48, ax=ax[0], color='#2b5c8f')
ax[0].set_title('Normal Transactions Over Time (Hours)', fontsize=12, fontweight='bold')
ax[0].set_ylabel('Transaction Count')

sns.histplot(credit_card_data[credit_card_data['Class'] == 1]['Time'] / 3600, bins=48, ax=ax[1], color='#d9534f')
ax[1].set_title('Fraudulent Transactions Over Time (Hours)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Time Elapsed (Hours)')
ax[1].set_ylabel('Transaction Count')

plt.tight_layout()
plt.show()

## Feature Correlation Analysis

Identifying which principal components correlate most strongly with fraudulent activity.

In [ ]:
# Correlation with the target Class
correlations = credit_card_data.corr()['Class'].sort_values()

print("Top 5 Negatively Correlated Features:")
print(correlations.head(5).round(4))

print("\nTop 5 Positively Correlated Features:")
print(correlations.tail(6).drop('Class').round(4))

In [ ]:
# Correlation heatmap of the top correlated features
top_features = list(correlations.head(5).index) + list(correlations.tail(6).index)
plt.figure(figsize=(10, 8))
sns.heatmap(credit_card_data[top_features].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Top Fraud Indicators', fontsize=13, fontweight='bold')
plt.show()

## Boxplots of Top Discriminating Features

In [ ]:
# Visualizing separating power of key features (V14, V12, V10, V4)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='Class', y='V14', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=axes[0, 0])
axes[0, 0].set_title('V14 vs Class (Strong Negative Correlation)')

sns.boxplot(x='Class', y='V12', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=axes[0, 1])
axes[0, 1].set_title('V12 vs Class (Strong Negative Correlation)')

sns.boxplot(x='Class', y='V17', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=axes[1, 0])
axes[1, 0].set_title('V17 vs Class (Strong Negative Correlation)')

sns.boxplot(x='Class', y='V4', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=axes[1, 1])
axes[1, 1].set_title('V4 vs Class (Strong Positive Correlation)')

for ax in axes.flat:
    ax.set_xticklabels(['Normal (0)', 'Fraud (1)'])

plt.tight_layout()
plt.show()

## Summary of Key EDA Findings

1. Extreme Imbalance: Only 492 frauds out of 284,807 transactions (0.172%). Baseline accuracy is not a reliable metric.
2. Key Signals: Features V14, V12, V10, V17 show strong negative shifts for fraud, while V4 and V11 show positive shifts.
3. Skewness: Transaction Amount is heavily skewed to the right. Log transformation is recommended.
4. Time: Non-linear temporal pattern; should be dropped from feature space to avoid sequence memorization.